In [1]:
from datasets import load_from_disk

DATASET_PATH = "/Volumes/Hardik/Arxiv_dataset/arxiv_50k_cleaned"

dataset = load_from_disk(DATASET_PATH)

train = dataset["train"]
validation = dataset["validation"]
test = dataset["test"]

print("Train:", len(train))
print("Validation:", len(validation))
print("Test:", len(test))

Train: 44971
Validation: 2500
Test: 2500


In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "allenai/led-large-16384-arxiv"

device = "mps" if torch.backends.mps.is_available() else "cpu"

print("Device:", device)

Device: mps


In [2]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Maximum input length:", tokenizer.model_max_length)

config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/27.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

Maximum input length: 16384


In [ ]:
from huggingface_hub import snapshot_download

MODEL_PATH = "/Volumes/Hardik/NLP_Project/models/summarizer/led-large-arxiv"

snapshot_download(
    repo_id="allenai/led-large-16384-arxiv",
    local_dir=MODEL_PATH
)

print("Model downloaded.")

In [1]:
from datasets import load_from_disk

DATASET_PATH = "/Volumes/Hardik/Arxiv_dataset/arxiv_50k_cleaned"

dataset = load_from_disk(DATASET_PATH)

train = dataset["train"]
validation = dataset["validation"]
test = dataset["test"]

print(len(train), len(validation), len(test))

44971 2500 2500


In [2]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

MODEL_PATH = "/Volumes/Hardik/NLP_Project/models/summarizer/led-large-arxiv"

device = "mps" if torch.backends.mps.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH)

model = model.to(device)

print("Device:", device)
print("Model loaded.")

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

Device: mps
Model loaded.


In [3]:
text = train[0]["article"]

inputs = tokenizer(
    text,
    max_length=16384,
    truncation=True,
    return_tensors="pt"
)

inputs = {key: value.to(device) for key, value in inputs.items()}

global_attention_mask = torch.zeros_like(inputs["input_ids"])
global_attention_mask[:, 0] = 1

with torch.no_grad():
    output = model.generate(
        **inputs,
        global_attention_mask=global_attention_mask,
        max_length=300,
        min_length=100,
        num_beams=4
    )

summary = tokenizer.decode(
    output[0],
    skip_special_tokens=True
)

print(summary)

[transformers] Input ids are automatically padded from 6338 to 7168 to be a multiple of `config.attention_window`: 1024


 arp 220 is the nearest example of an ultraluminous infrared galaxy ( ulirg ) that supports star formation at extreme levels . 
 radio detections of supernovae at a rate of 13 yr [ mATH] confirm that huge populations of massive stars are present with an implied star formation rate ( sfr ) of [ mATH] yr [ mATH] . in this paper , we study cosmic ray interactions in the arp 220 starburst nuclear regions using an updated version of the models we develop a model with two spatial zones to accurately represent the inner and outer regions of the western nucleus defined by its molecular gas properties we incorporate energy losses and photon interactions to account for the extreme calorimetry of the nuclear starburst zones . 
 we find that milligauss strength magnetic fields are still necessary to reproduce the observed radio fluxes from the starburst nuclei even having assumed larger supernova rates than previous models by factors of 2 5. differences in assumed volume and radiation field energy

In [4]:
def led_summarize(text, max_input_tokens=16000, max_output_tokens=300):
    inputs = tokenizer(
        text,
        max_length=max_input_tokens,
        truncation=True,
        return_tensors="pt"
    )

    inputs = {key: value.to(device) for key, value in inputs.items()}

    global_attention_mask = torch.zeros_like(inputs["input_ids"])
    global_attention_mask[:, 0] = 1

    with torch.no_grad():
        output = model.generate(
            **inputs,
            global_attention_mask=global_attention_mask,
            max_length=max_output_tokens,
            min_length=100,
            num_beams=4
        )

    return tokenizer.decode(output[0], skip_special_tokens=True)

In [5]:
summary = led_summarize(train[0]["article"])

print(summary)

 arp 220 is the nearest example of an ultraluminous infrared galaxy ( ulirg ) that supports star formation at extreme levels . 
 radio detections of supernovae at a rate of 13 yr [ mATH] confirm that huge populations of massive stars are present with an implied star formation rate ( sfr ) of [ mATH] yr [ mATH] . in this paper , we study cosmic ray interactions in the arp 220 starburst nuclear regions using an updated version of the models we develop a model with two spatial zones to accurately represent the inner and outer regions of the western nucleus defined by its molecular gas properties we incorporate energy losses and photon interactions to account for the extreme calorimetry of the nuclear starburst zones . 
 we find that milligauss strength magnetic fields are still necessary to reproduce the observed radio fluxes from the starburst nuclei even having assumed larger supernova rates than previous models by factors of 2 5. differences in assumed volume and radiation field energy

In [6]:
def split_into_chunks(text, max_tokens=12000):
    tokens = tokenizer.encode(text, add_special_tokens=False)

    chunks = []

    for i in range(0, len(tokens), max_tokens):
        chunk_tokens = tokens[i:i + max_tokens]
        chunks.append(
            tokenizer.decode(chunk_tokens, skip_special_tokens=True)
        )

    return chunks

In [7]:
text = train[0]["article"]

chunks = split_into_chunks(text)

print("Number of chunks:", len(chunks))

for i, chunk in enumerate(chunks):
    print(
        f"Chunk {i + 1}:",
        len(tokenizer.encode(chunk, add_special_tokens=False)),
        "tokens"
    )

Number of chunks: 1
Chunk 1: 6336 tokens


In [8]:
longest_index = max(
    range(len(train)),
    key=lambda i: len(train[i]["article"])
)

longest_text = train[longest_index]["article"]

print("Longest article characters:", len(longest_text))
print("Longest article tokens:", len(
    tokenizer.encode(longest_text, add_special_tokens=False)
))

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (130598 > 16384). Running this sequence through the model will result in indexing errors


Longest article characters: 452745
Longest article tokens: 130598


In [ ]:
def hierarchical_summarize(text):
    chunks = split_into_chunks(text, max_tokens=12000)

    chunk_summaries = []

    for i, chunk in enumerate(chunks):
        print(f"Summarizing chunk {i + 1}/{len(chunks)}")
        summary = led_summarize(
            chunk,
            max_input_tokens=12000,
            max_output_tokens=300
        )
        chunk_summaries.append(summary)

    combined = " ".join(chunk_summaries)

    while len(tokenizer.encode(combined, add_special_tokens=False)) > 12000:
        combined_tokens = tokenizer.encode(
            combined,
            add_special_tokens=False
        )

        midpoint = len(combined_tokens) // 2

        part1 = tokenizer.decode(
            combined_tokens[:midpoint],
            skip_special_tokens=True
        )

        part2 = tokenizer.decode(
            combined_tokens[midpoint:],
            skip_special_tokens=True
        )

        combined = (
            led_summarize(part1, max_output_tokens=300)
            + " "
            + led_summarize(part2, max_output_tokens=300)
        )

    return led_summarize(
        combined,
        max_input_tokens=12000,
        max_output_tokens=300
    )